# DataCoolie Databricks Spark Sample

Run DataCoolie file + delta scenarios on Databricks using `DatabricksPlatform` and file metadata (`databricks_use_cases.json`).

Recommended first run:
- `STAGE = "read_file,load_delta"`

Then run all file/delta scenarios:
- `STAGE = ""`

In [ ]:
%pip install "datacoolie" --upgrade
# Optional local wheel install:
# %pip install /Volumes/workspace/default/datacoolie_sim/libraries/datacoolie-0.1.0-py3-none-any.whl --force-reinstall

In [ ]:
dbutils.library.restartPython()  # type: ignore[name-defined]

In [ ]:
from datacoolie.core.models import DataCoolieRunConfig
from datacoolie.engines.spark_engine import SparkEngine
from datacoolie.metadata.file_provider import FileProvider
from datacoolie.orchestration.driver import DataCoolieDriver
from datacoolie.platforms.databricks_platform import DatabricksPlatform

# Replace with your UC volume root if needed.
VOLUME_ROOT = "/Volumes/workspace/default/datacoolie_sim"
METADATA_PATH = f"{VOLUME_ROOT}/metadata/databricks_use_cases.json"
BASE_LOG_PATH = f"{VOLUME_ROOT}/logs/datacoolie"

# Narrow first run for smoke validation. Use STAGE="" to run all file+delta scenarios.
STAGE = "read_file,load_delta"

platform = DatabricksPlatform()
engine = SparkEngine(spark, platform=platform)  # spark is provided by Databricks runtime
metadata = FileProvider(config_path=METADATA_PATH, platform=platform)
run_config = DataCoolieRunConfig(max_workers=8)

with DataCoolieDriver(
    engine=engine,
    metadata_provider=metadata,
    base_log_path=BASE_LOG_PATH,
    config=run_config,
) as driver:
    result = driver.run(stage=STAGE)

print(
    "Result:",
    {
        "total": result.total,
        "succeeded": result.succeeded,
        "failed": result.failed,
        "skipped": result.skipped,
    },
)